In [0]:
bronze_df=spark.read.table('bronze.bronze_sales_delta')    
bronze_df.show()

In [0]:
from pyspark.sql import functions as F
df = bronze_df

columns = [
    "sales_order_id",
    "customer_id",
    "material_id",
    "order_date",
    "quantity",
    "unit_price"
]

for c in columns:
    value = F.trim(F.col(c).cast("string"))

    df = df.withColumn(
        c,
        F.when(
            F.upper(value).isin("", "NULL", "NONE", "N/A"),
            F.lit(None)
        ).otherwise(value)
    )

In [0]:
df = (
    df
    .withColumn(
        "order_date_clean",
        F.expr("try_cast(order_date AS DATE)")
    )
    .withColumn(
        "quantity_clean",
        F.expr("try_cast(quantity AS INT)")
    )
    .withColumn(
        "unit_price_clean",
        F.expr("try_cast(unit_price AS DECIMAL(18,2))")
    )
)

In [0]:
validated_df = df.withColumn(
    "reject_reason",
    F.concat_ws(
        ", ",
        F.when(
            F.col("sales_order_id").isNull(),
            "MISSING_SALES_ORDER_ID"
        ),
        F.when(
            F.col("customer_id").isNull(),
            "MISSING_CUSTOMER_ID"
        ),
        F.when(
            F.col("material_id").isNull(),
            "MISSING_MATERIAL_ID"
        ),
        F.when(
            F.col("order_date").isNull(),
            "MISSING_ORDER_DATE"
        ),
        F.when(
            F.col("order_date").isNotNull()
            & F.col("order_date_clean").isNull(),
            "INVALID_ORDER_DATE"
        ),
        F.when(
            F.col("quantity").isNull(),
            "MISSING_QUANTITY"
        ),
        F.when(
            F.col("quantity").isNotNull()
            & F.col("quantity_clean").isNull(),
            "INVALID_QUANTITY"
        ),
        F.when(
            F.col("quantity_clean") <= 0,
            "NON_POSITIVE_QUANTITY"
        ),
        F.when(
            F.col("unit_price_clean").isNull()
            | (F.col("unit_price_clean") <= 0),
            "INVALID_UNIT_PRICE"
        )
    )
)

display(validated_df)

In [0]:
valid_df = validated_df.filter(
    F.col("reject_reason") == ""
)

rejected_df = validated_df.filter(
    F.col("reject_reason") != ""
)

print("Valid:", valid_df.count())
print("Rejected:", rejected_df.count())

In [0]:
silver_df = valid_df.select(
    "sales_order_id",
    "customer_id",
    "material_id",
    F.col("order_date_clean").alias("order_date"),
    F.col("quantity_clean").alias("quantity"),
    F.col("unit_price_clean").alias("unit_price"),
    (
        F.col("quantity_clean")
        * F.col("unit_price_clean")
    ).cast("decimal(18,2)").alias("order_amount"),
    "source_file",
    "ingestion_timestamp"
)

display(silver_df)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "wfrgbhn.silver.silver_salesorders_delta"
    )

rejected_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "wfrgbhn.silver.reject_sales_orders_delta"
    )

In [0]:
import pyspark.sql.functions as F
gold_material_df = (
    silver_df
    .groupBy("material_id")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.sum("order_amount").alias("total_revenue"),
        F.countDistinct("sales_order_id").alias("total_orders")
    )
    .orderBy(F.col("total_revenue").desc())
)

display(gold_material_df)

In [0]:
gold_material_df.write.mode("overwrite").saveAsTable("wfrgbhn.gold.gold_materials_delta")